# Notebook 2 · Who is growing? (Question 2)

**Question:** Which SBLCs lead on volume, and who gained or lost ground over the decade?

**Approach**
1. **Market context:** is the SBLC segment growing faster than the overall 7(a) market?
2. **League table:** total volume by lender over FY2010-2019.
3. **Share gained or lost:** compare FY2010-12 with FY2017-19, using 3-year windows to smooth lumpy years.
4. **Concentration:** is the SBLC group becoming dominated by a few lenders?
5. **Why:** did lenders change their business model (loan size, product)?
6. **Real competitors:** which SBLCs look most alike?

**Metric definitions**
- **Volume** = gross approval dollars (and number of loans) by approval fiscal year.
- **Market share** = lender dollars / all 7(a) dollars in the same period, in basis points (1 bp = 0.01%). Using share, not raw dollars, removes the effect of a growing market.
- **Pool share** = lender dollars / all SBLC dollars. This shows who wins *within* the SBLC group.

**Inputs:** `data/processed/loans_clean.parquet` (Notebook 0), `assignment1/outputs/sblc_list.csv` (Notebook 1)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

In [4]:
df = pd.read_parquet("../../data/processed/loans_clean.parquet")
sblc = pd.read_csv("../../assignment1/outputs/sblc_list.csv")
print("Loans:", len(df), "| SBLCs from Notebook 1:", len(sblc))

Loans: 478650 | SBLCs from Notebook 1: 12


## Step 0 · Flag SBLC loans

We use the 12 SBLCs from Notebook 1 (tiers 1-Confirmed and 2-Likely). Short names are used in charts.

In [5]:
SHORT = {
    "Newtek Small Business Finance, Inc.": "Newtek",
    "VelocitySBA, LLC": "Velocity",
    "Readycap Lending, LLC": "Readycap",
    "Hana Small Business Lending, Inc.": "Hana",
    "Harvest Small Business Finance, LLC": "Harvest",
    "First Western SBLC, Inc": "First Western",
    "CRF Small Business Loan Company, LLC": "CRF",
    "Centerstone SBA Lending, Inc.": "Centerstone",
    "Grow America Fund, Incorporated": "Grow America",
    "Lendistry SBLC, LLC": "Lendistry",
    "Loan Source Incorporated": "Loan Source",
    "CapitalSpring SBLC, LLC": "CapitalSpring",
}

# Safety check: the names above must match the list from Notebook 1 exactly
assert set(SHORT) == set(sblc["bankname"]), "SBLC list changed: update SHORT"

df["is_sblc"] = df["bankname"].isin(SHORT)
df["lender"] = df["bankname"].map(SHORT)
print("SBLC loans:", df["is_sblc"].sum(), f"({df['is_sblc'].mean():.1%} of all clean loans)")

SBLC loans: 11465 (2.4% of all clean loans)


## Step 1 · Market context: is the SBLC segment growing faster than the market?
For each fiscal year: total 7(a) approvals, SBLC approvals, and the SBLC share.

In [6]:
mkt = df.groupby("approvalfy").agg(mkt_loans=("grossapproval", "size"), mkt_gross=("grossapproval", "sum"))
sb = df[df["is_sblc"]].groupby("approvalfy").agg(sblc_loans=("grossapproval", "size"), sblc_gross=("grossapproval", "sum"))
seg = mkt.join(sb)
seg["sblc_loan_share_%"] = seg["sblc_loans"] / seg["mkt_loans"] * 100
seg["sblc_gross_share_%"] = seg["sblc_gross"] / seg["mkt_gross"] * 100
display(seg.round(2))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=seg.index, y=seg["sblc_gross"] / 1e9, name="SBLC approvals ($bn)")
fig.add_scatter(x=seg.index, y=seg["sblc_gross_share_%"], name="SBLC share of 7(a) $ (%)",
                mode="lines+markers", secondary_y=True)
fig.update_xaxes(type="category", title_text="Approval fiscal year")
fig.update_yaxes(title_text="$bn", secondary_y=False)
fig.update_yaxes(title_text="% of all 7(a) dollars", secondary_y=True, rangemode="tozero")
fig.update_layout(title="SBLC approvals and market share", template="plotly_white", height=420)
fig.show()

,mkt_loans,mkt_gross,sblc_loans,sblc_gross,sblc_loan_share_%,sblc_gross_share_%
approvalfy,,,,,,
2010,39909,10184745700,3481,300477800,8.72,2.95
2011,45586,16201937800,1799,433301000,3.95,2.67
2012,38854,13220237700,517,336826800,1.33,2.55
2013,40385,15454865600,580,367126300,1.44,2.38
2014,45908,16844614700,492,415319600,1.07,2.47
2015,55372,20314682100,554,488137000,1.00,2.40
2016,56745,21595305443,718,568333700,1.27,2.63
2017,56062,23047665800,891,855781800,1.59,3.71
2018,54172,22807258700,1249,1152223900,2.31,5.05


## Step 2 · League table and yearly volume
Who lends the most over the whole decade, and how did each lender's yearly volume move?

In [7]:
tot = (df[df["is_sblc"]].groupby("lender")
       .agg(loans=("grossapproval", "size"),
            gross_usd_m=("grossapproval", lambda x: x.sum() / 1e6),
            median_loan=("grossapproval", "median"))
       .sort_values("gross_usd_m", ascending=False))
tot["pool_share_%"] = tot["gross_usd_m"] / tot["gross_usd_m"].sum() * 100
display(tot.round(1))

ly = (df[df["is_sblc"]].groupby(["lender", "approvalfy"])
      .agg(loans=("grossapproval", "size"), gross_usd_m=("grossapproval", lambda x: x.sum() / 1e6))
      .reset_index())

fig = px.bar(ly, x="approvalfy", y="gross_usd_m", color="lender",
             category_orders={"lender": tot.index.tolist()},
             labels={"approvalfy": "Approval fiscal year", "gross_usd_m": "Gross approvals ($m)"},
             title="SBLC approvals by lender and year")
fig.update_xaxes(type="category")
fig.update_layout(template="plotly_white", height=450)
fig.show()

,loans,gross_usd_m,median_loan,pool_share_%
lender,,,,
Newtek,2874,2445.3,405500.0,40.9
Readycap,975,963.5,662000.0,16.1
Hana,671,624.4,528000.0,10.4
Harvest,478,607.8,927300.0,10.2
First Western,295,462.6,1364000.0,7.7
Centerstone,205,253.6,850000.0,4.2
CRF,494,247.4,300000.0,4.1
Grow America,268,154.2,315000.0,2.6
Velocity,5008,130.9,10000.0,2.2


## Step 3 · Who gained or lost ground?
We compare two 3-year windows: **early = FY2010-12** and **late = FY2017-19**. Three years smooth out lumpy single years.

For each lender:
- **Market share (bps):** its dollars / all 7(a) dollars in the window.
- **Share change:** late minus early. This is the clearest gain or loss, because it removes the growth of the whole market.
- **Pool share (%):** its dollars / all SBLC dollars in the window.

In [8]:
EARLY, LATE = (2010, 2012), (2017, 2019)

def window(lo, hi):
    """Lender totals and market totals for approval years lo..hi."""
    w = df[df["approvalfy"].between(lo, hi)]
    by_lender = w[w["is_sblc"]].groupby("lender").agg(loans=("grossapproval", "size"), gross=("grossapproval", "sum"))
    return by_lender, w["grossapproval"].sum()

e, e_mkt = window(*EARLY)
l, l_mkt = window(*LATE)

gr = pd.DataFrame({
    "gross_early_m": e["gross"] / 1e6, "gross_late_m": l["gross"] / 1e6,
    "loans_early": e["loans"], "loans_late": l["loans"],
}).reindex(tot.index).fillna(0)

gr["share_early_bps"] = gr["gross_early_m"] * 1e6 / e_mkt * 1e4
gr["share_late_bps"] = gr["gross_late_m"] * 1e6 / l_mkt * 1e4
gr["share_change_bps"] = gr["share_late_bps"] - gr["share_early_bps"]
gr["pool_share_early_%"] = gr["gross_early_m"] / gr["gross_early_m"].sum() * 100
gr["pool_share_late_%"] = gr["gross_late_m"] / gr["gross_late_m"].sum() * 100
gr["rank_early"] = gr["gross_early_m"].rank(ascending=False, method="min")
gr["rank_late"] = gr["gross_late_m"].rank(ascending=False, method="min")

print(f"Whole 7(a) market grew {l_mkt / e_mkt:.2f}x from early to late window")
print(f"SBLC group grew {gr['gross_late_m'].sum() / gr['gross_early_m'].sum():.2f}x "
      f"(${gr['gross_early_m'].sum():,.0f}m -> ${gr['gross_late_m'].sum():,.0f}m)")
gr.round(1)

Whole 7(a) market grew 1.68x from early to late window
SBLC group grew 2.87x ($1,071m -> $3,068m)


,gross_early_m,gross_late_m,loans_early,loans_late,share_early_bps,share_late_bps,share_change_bps,pool_share_early_%,pool_share_late_%,rank_early,rank_late
lender,,,,,,,,,,,
Newtek,260.2,1264.7,302.0,1573.0,65.7,190.4,124.7,24.3,41.2,2.0,1.0
Readycap,291.0,540.3,320.0,530.0,73.5,81.4,7.9,27.2,17.6,1.0,3.0
Hana,247.7,46.4,276.0,48.0,62.5,7.0,-55.5,23.1,1.5,3.0,8.0
Harvest,43.3,554.0,44.0,424.0,10.9,83.4,72.5,4.0,18.1,6.0,2.0
First Western,107.2,189.6,98.0,91.0,27.1,28.6,1.5,10.0,6.2,4.0,5.0
Centerstone,0.0,234.9,0.0,183.0,0.0,35.4,35.4,0.0,7.7,11.0,4.0
CRF,14.1,84.7,22.0,177.0,3.6,12.8,9.2,1.3,2.8,9.0,6.0
Grow America,37.9,44.6,75.0,86.0,9.6,6.7,-2.8,3.5,1.5,7.0,9.0
Velocity,48.7,75.7,4643.0,67.0,12.3,11.4,-0.9,4.6,2.5,5.0,7.0


In [9]:
sc = gr.sort_values("share_change_bps")
fig = px.bar(sc, x="share_change_bps", y=sc.index, orientation="h",
             color=np.where(sc["share_change_bps"] >= 0, "Gained share", "Lost share"),
             color_discrete_map={"Gained share": "#2a9d8f", "Lost share": "#e76f51"},
             labels={"share_change_bps": "Change in share of all 7(a) dollars (bps), FY2010-12 to FY2017-19",
                     "y": "", "color": ""},
             title="Who gained and who lost ground")
fig.update_layout(template="plotly_white", height=450)
fig.show()

### Ranking inside the SBLC group
The slope chart shows each lender's share of SBLC dollars early versus late. Lines going up gained ground against the other SBLCs.

In [10]:
fig = go.Figure()
for name, r in gr.iterrows():
    fig.add_scatter(x=["FY2010-12", "FY2017-19"],
                    y=[r["pool_share_early_%"], r["pool_share_late_%"]],
                    mode="lines+markers+text", name=name,
                    text=[name, f"{name} {r['pool_share_late_%']:.0f}%"],
                    textposition=["middle left", "middle right"], textfont_size=10)
fig.update_layout(title="Share of SBLC dollars, early vs late", yaxis_title="% of SBLC approvals",
                  showlegend=False, template="plotly_white", height=500)
fig.show()

gr[["rank_early", "rank_late", "pool_share_early_%", "pool_share_late_%"]].round(1)

,rank_early,rank_late,pool_share_early_%,pool_share_late_%
lender,,,,
Newtek,2.0,1.0,24.3,41.2
Readycap,1.0,3.0,27.2,17.6
Hana,3.0,8.0,23.1,1.5
Harvest,6.0,2.0,4.0,18.1
First Western,4.0,5.0,10.0,6.2
Centerstone,11.0,4.0,0.0,7.7
CRF,9.0,6.0,1.3,2.8
Grow America,7.0,9.0,3.5,1.5
Velocity,5.0,7.0,4.6,2.5


## Step 4 · Is the SBLC group getting more concentrated?
The **HHI** (Herfindahl-Hirschman index) adds up each lender's squared share of the group. It runs from near 0 (many equal players) to 10,000 (one player). A rising HHI means a few lenders are taking over.

In [11]:
def hhi(s):
    return ((s / s.sum() * 100) ** 2).sum()

print(f"HHI early: {hhi(gr['gross_early_m']):,.0f}   |   HHI late: {hhi(gr['gross_late_m']):,.0f}")
top2 = gr["gross_late_m"].nlargest(2)
print(f"Top 2 lenders in late window: {', '.join(top2.index)} = {top2.sum() / gr['gross_late_m'].sum():.0%} of SBLC dollars")

HHI early: 2,019   |   HHI late: 2,451
Top 2 lenders in late window: Newtek, Harvest = 59% of SBLC dollars


## Step 5 · Why? Did lenders change their business model?
A lender can gain share by lending more of the same thing, or by switching what it lends. We compare each lender's **median loan size** and **product mix** between the two windows:
- **PLP share:** the Preferred Lenders Program, used by established commercial lenders.
- **Express share:** the SBA Express programs, which are for smaller, faster loans.

In [12]:
df["period"] = np.where(df["approvalfy"].between(*EARLY), "Early",
                np.where(df["approvalfy"].between(*LATE), "Late", "Mid"))
two = df[df["is_sblc"] & df["period"].isin(["Early", "Late"])]

bm = two.groupby(["lender", "period"]).agg(
    loans=("grossapproval", "size"),
    median_loan=("grossapproval", "median"),
    plp_share=("processingmethod", lambda s: s.isin(["Preferred Lenders Program", "Preferred Lenders with EWCP"]).mean()),
    express_share=("processingmethod", lambda s: s.str.contains("Express", na=False).mean()),
).unstack("period")
display(bm.reindex(tot.index).round(2))

med = two.groupby(["lender", "period"])["grossapproval"].median().reset_index()
fig = px.bar(med, x="lender", y="grossapproval", color="period", barmode="group", log_y=True,
             category_orders={"lender": tot.index.tolist(), "period": ["Early", "Late"]},
             labels={"grossapproval": "Median loan ($, log scale)", "lender": ""},
             title="Median loan size: early vs late")
fig.update_layout(template="plotly_white", height=420)
fig.show()

loans         median_loan            plp_share       express_share      
period          Early    Late       Early       Late     Early  Late         Early  Late
lender                                                                                  
Newtek          302.0  1573.0    550000.0   365000.0      0.99  0.96          0.00  0.02
Readycap        320.0   530.0    594500.0   715500.0      0.71  0.93          0.00  0.00
Hana            276.0    48.0    500000.0   489500.0      1.00  0.85          0.00  0.00
Harvest          44.0   424.0    762500.0   933300.0      0.50  0.94          0.00  0.00
First Western    98.0    91.0   1011500.0  1905000.0      0.88  0.92          0.00  0.00
Centerstone       NaN   183.0         NaN   925000.0       NaN  0.90           NaN  0.00
CRF              22.0   177.0    478750.0   320000.0      0.00  0.89          0.00  0.00
Grow America     75.0    86.0    315000.0   272500.0      1.00  0.63          0.00  0.00
Velocity       4643.0    67.0     10000.0   788000.0      0.00  0.63          0.99  0.00
Loan Source      16.0     5.0    925000.0   540000.0      0.88  1.00          0.00  0.00
Lendistry         NaN   140.0         NaN   219000.0       NaN  0.01           NaN  0.00
CapitalSpring     1.0     NaN   3310000.0        NaN      0.00   NaN          0.00   NaN

## Step 6 · Who are the *real* competitors?
Two SBLCs compete only if they serve similar customers. We describe each lender with six measures: median loan size (log), PLP share, Express share, number of states, average term and the share of new or startup businesses. We standardise them and measure the distance between lenders. **Small distance = similar business model.**

Caveat: the profile uses the whole decade, so a lender that changed its model (for example Velocity) shows a blend of its old and new business.

In [13]:
s = df[df["is_sblc"]]
g = s.groupby("lender")

prof = pd.DataFrame({
    "median_loan": g["grossapproval"].median(),
    "plp_share": g["processingmethod"].apply(lambda x: x.isin(["Preferred Lenders Program", "Preferred Lenders with EWCP"]).mean()),
    "express_share": g["processingmethod"].apply(lambda x: x.str.contains("Express", na=False).mean()),
    "n_states": g["borrstate"].nunique(),
    "top_state_share": g["borrstate"].apply(lambda x: x.value_counts(normalize=True).iloc[0]),
    "collateral_share": g["collateralind"].apply(lambda x: (x.astype(str) == "True").mean()),
    "avg_term_m": g["terminmonths"].mean(),
    "startup_share": g["businessage"].apply(lambda x: x.astype(str).str.contains("Startup|New", case=False).mean()),
    "sold_secondary": g["soldsecmrktind"].apply(lambda x: (x == "Y").mean()),
})
prof["log_loan"] = np.log10(prof["median_loan"])
display(prof.reindex(tot.index).round(2))

# Standardise the six features and compute distance between every pair of lenders
features = ["log_loan", "plp_share", "express_share", "n_states", "avg_term_m", "startup_share"]
z = ((prof[features] - prof[features].mean()) / prof[features].std(ddof=0)).fillna(0)
dist = pd.DataFrame(np.linalg.norm(z.values[:, None] - z.values[None, :], axis=2),
                    index=z.index, columns=z.index)

nearest = dist.apply(lambda r: ", ".join(r.drop(r.name).nsmallest(2).index), axis=1)
display(nearest.to_frame("two closest peers"))

top_ind = s.groupby("lender")["naicsdescription"].apply(
    lambda x: ", ".join(x.str.strip().value_counts().head(2).index))
display(top_ind.to_frame("top 2 industries"))

fig = px.imshow(dist.loc[tot.index, tot.index].round(2), color_continuous_scale="Blues_r",
                text_auto=True, aspect="auto", title="Distance between SBLC business models (lower = more alike)")
fig.update_layout(height=550)
fig.show()

,median_loan,plp_share,express_share,n_states,top_state_share,collateral_share,avg_term_m,startup_share,sold_secondary,log_loan
lender,,,,,,,,,,
Newtek,405500.0,0.91,0.01,52,0.12,0.96,167.46,0.11,0.99,5.61
Readycap,662000.0,0.82,0.00,47,0.15,0.87,214.15,0.27,0.99,5.82
Hana,528000.0,0.86,0.00,20,0.59,0.99,196.34,0.15,1.00,5.72
Harvest,927300.0,0.88,0.00,34,0.63,0.99,291.75,0.04,1.00,5.97
First Western,1364000.0,0.88,0.00,34,0.13,0.99,290.54,0.05,1.00,6.13
Centerstone,850000.0,0.80,0.00,14,0.80,1.00,194.01,0.09,1.00,5.93
CRF,300000.0,0.67,0.00,43,0.17,0.99,138.94,0.48,0.97,5.48
Grow America,315000.0,0.85,0.00,14,0.28,0.99,169.93,0.03,0.57,5.50
Velocity,10000.0,0.01,0.94,51,0.13,0.02,107.39,0.24,0.99,4.00


,two closest peers
lender,
CRF,"Readycap, CapitalSpring"
CapitalSpring,"Lendistry, Hana"
Centerstone,"Hana, Grow America"
First Western,"Harvest, Centerstone"
Grow America,"Centerstone, Hana"
Hana,"Centerstone, Grow America"
Harvest,"First Western, Hana"
Lendistry,"CapitalSpring, Grow America"
Loan Source,"Hana, Centerstone"


,top 2 industries
lender,
CRF,"Limited-Service Restaurants, Full-Service Rest..."
CapitalSpring,"Limited-Service Restaurants, Full-Service Rest..."
Centerstone,"Beer, Wine, and Liquor Stores, Full-Service Re..."
First Western,"Hotels (except Casino Hotels) and Motels, RV (..."
Grow America,"Full-Service Restaurants, All Other Specialty ..."
Hana,"Full-Service Restaurants, Hotels (except Casin..."
Harvest,"Full-Service Restaurants, Hotels (except Casin..."
Lendistry,"Limited-Service Restaurants, Full-Service Rest..."
Loan Source,"Full-Service Restaurants, Offices of Physical,..."


## Save outputs for later notebooks

In [14]:
out = Path("../../assignment1/outputs"); out.mkdir(parents=True, exist_ok=True)
seg.to_csv(out / "sblc_market_by_year.csv")
ly.to_csv(out / "sblc_lender_year.csv", index=False)
gr.to_csv(out / "sblc_growth_windows.csv")
prof.to_csv(out / "sblc_profile.csv")
print("Saved:", sorted(p.name for p in out.glob("sblc_*.csv")))

Saved: ['sblc_candidates_scored.csv', 'sblc_growth_windows.csv', 'sblc_lender_year.csv', 'sblc_list.csv', 'sblc_market_by_year.csv', 'sblc_profile.csv']


## Summary: who is growing?

**The SBLC segment is outgrowing the market.** Between FY2010-12 and FY2017-19, 7(a) approvals grew 1.7x, but SBLC approvals grew 2.9x (about $1.1bn to $3.1bn). SBLCs went from roughly 2.4-3% of all 7(a) dollars up to about 5% by FY2018-19.

**Winners**
- **Newtek** leads. Its share of all 7(a) dollars rose from about 66 to 190 bps, and it holds about 41% of SBLC dollars (up from 24%).
- **Harvest** rose from about 11 to 83 bps, mostly from FY2017 onwards.
- **Centerstone** entered in FY2016 and reached about 35 bps.
- **CRF** is a smaller gainer (about +9 bps).

**Losers and flat**
- **Hana** lost the most ground (about 63 to 7 bps) and had almost stopped by FY2018-19.
- **Readycap** held its market share (+8 bps) but lost ground *inside* the SBLC group, as Newtek and Harvest grew faster (pool share about 27% to 18%).
- Grow America, Loan Source and CapitalSpring are small and drifting down.

**Concentration is rising.** The HHI went from about 2,000 to about 2,450, and Newtek plus Readycap hold about 59% of late-window dollars.

**Why they differ: business models changed.**
- **Velocity** switched from about 4,600 tiny loans (median about $10k, almost all Express) to about 70 large PLP loans (median about $790k). Its share looks flat in dollars, but it is a different company.
- **CRF** moved from almost no PLP loans to about 90%.
- **Peers:** Newtek and Readycap look alike (national, broad). Harvest and First Western look alike (large ticket, concentrated). Hana, Centerstone and Grow America cluster together.

**Caveats**
- Small lenders (fewer than 300 loans) have lumpy yearly numbers, so we judge them over 3-year windows.
- Share is measured on approvals, not on loans that actually performed. Growth quality is Notebook 3.
- Newtek converted to a bank after FY2019, so its later history is outside this data.